# Grid: route choice under congestion

All agents travel from the bottom-left to the top-right corner of a grid. With low link capacities they
learn to spread over the many equally long routes. This demo also shows how to compose a custom
strategy mix.

In [ ]:
import logging

from mobslim import Controller, ControllerParams, EventRecorder, Plan, Population, Scenario, grid_network
from mobslim.animate import animate_events
from mobslim.replanning import ChangeExpBetaSelector, PlanStrategy, RandomSelector, Reroute, StrategyManager

logging.basicConfig(level=logging.INFO, format="%(message)s")

AGENTS, SIZE, ITERATIONS = 20, 4, 15

In [ ]:
network = grid_network(size=SIZE, capacity=360)  # 1 vehicle / 10 s per link
origin, destination = network.node_index((0, 0)), network.node_index((SIZE, SIZE))

plans = {
    i: Plan.builder().act("home", origin, end_time=0).leg("car").act("work", destination).build()
    for i in range(AGENTS)
}
scenario = Scenario(network, Population.from_plans(plans))

In [ ]:
strategies = StrategyManager(
    [
        (0.7, PlanStrategy(ChangeExpBetaSelector())),
        (0.3, PlanStrategy(RandomSelector(), [Reroute()])),
    ],
    max_plans=4,
    disable_innovation_at=int(0.8 * ITERATIONS),
)
recorder = EventRecorder()
result = Controller(
    scenario, strategies=strategies, listeners=[recorder], params=ControllerParams(iterations=ITERATIONS)
).run()
result.metrics_frame()[["score_executed", "car_travel_time"]]

In [ ]:
animate_events(network, recorder.last, step=2)